# Deep Shipping Traffic Analysis
Investigating ship types, periods of disruption, and sentiment triggers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

plt.rcParams['figure.figsize'] = (18, 10)
plt.rcParams['font.size'] = 11

## Load and Prepare Data

In [ ]:
arrivals_df = pd.read_csv('Data/Portwatch_Shipment_Data/arrivals-of-ships.csv')
arrivals_df['DateTime'] = pd.to_datetime(arrivals_df['DateTime'])
arrivals_df.set_index('DateTime', inplace=True)

arrivals_df['Total'] = arrivals_df[['Container', 'Dry Bulk', 'General Cargo', 'Roll-on/roll-off', 'Tanker']].sum(axis=1)

arrivals_df.head()

In [ ]:
arrivals_df.info()

In [ ]:
arrivals_df.describe()

## Define Crisis Periods and Sentiment
Based on known geopolitical events affecting shipping

In [ ]:
crisis_periods = [
    {
        'start': '2019-05-12',
        'end': '2019-07-31',
        'event': 'Gulf of Oman Tanker Attacks',
        'sentiment': 'High Tension',
        'severity': 'High'
    },
    {
        'start': '2020-01-03',
        'end': '2020-02-15',
        'event': 'US-Iran Escalation (Soleimani)',
        'sentiment': 'Extreme Tension',
        'severity': 'Critical'
    },
    {
        'start': '2021-04-11',
        'end': '2021-05-20',
        'event': 'Israel-Iran Naval Incidents',
        'sentiment': 'Elevated Tension',
        'severity': 'Medium'
    },
    {
        'start': '2022-02-24',
        'end': '2022-04-30',
        'event': 'Ukraine War Impact on Energy',
        'sentiment': 'Supply Concerns',
        'severity': 'High'
    },
    {
        'start': '2023-10-07',
        'end': '2024-01-31',
        'event': 'Red Sea / Bab al-Mandab Crisis',
        'sentiment': 'Critical Disruption',
        'severity': 'Critical'
    },
    {
        'start': '2024-04-13',
        'end': '2024-05-15',
        'event': 'Iran-Israel Direct Exchange',
        'sentiment': 'Extreme Tension',
        'severity': 'Critical'
    },
    {
        'start': '2025-02-15',
        'end': '2025-02-25',
        'event': 'Hormuz Naval "Inspections" Warning',
        'sentiment': 'Market Signaling',
        'severity': 'Medium'
    },
    {
        'start': '2025-06-13',
        'end': '2025-06-24',
        'event': '12-Day Air War (US/ISR vs Iran)',
        'sentiment': 'Open State Warfare',
        'severity': 'Critical'
    },
    {
        'start': '2026-02-28',
        'end': '2026-03-06',
        'event': 'Operation Epic Fury / Full Blockade',
        'sentiment': 'Total Supply Shock',
        'severity': 'Critical'
    }
]

crisis_df = pd.DataFrame(crisis_periods)
crisis_df['start'] = pd.to_datetime(crisis_df['start'])
crisis_df['end'] = pd.to_datetime(crisis_df['end'])

# Display the full historical and contemporary risk timeline
crisis_df.head(10)

In [ ]:
def classify_period(date):
    for _, crisis in crisis_df.iterrows():
        if crisis['start'] <= date <= crisis['end']:
            return crisis['event'], crisis['sentiment'], crisis['severity']
    return 'Normal', 'Stable', 'Normal'

arrivals_df[['period', 'sentiment', 'severity']] = arrivals_df.index.to_series().apply(
    lambda x: pd.Series(classify_period(x))
)

arrivals_df['is_crisis'] = arrivals_df['severity'] != 'Normal'

arrivals_df.head(20)

## Overall Traffic Patterns: Normal vs Crisis

In [ ]:
normal_stats = arrivals_df[~arrivals_df['is_crisis']].describe()
crisis_stats = arrivals_df[arrivals_df['is_crisis']].describe()

comparison = pd.DataFrame({
    'Normal_Mean': normal_stats.loc['mean'],
    'Crisis_Mean': crisis_stats.loc['mean'],
    'Pct_Change': ((crisis_stats.loc['mean'] - normal_stats.loc['mean']) / normal_stats.loc['mean'] * 100)
})

comparison = comparison[['Normal_Mean', 'Crisis_Mean', 'Pct_Change']].round(2)
comparison

In [ ]:
ship_types = ['Container', 'Dry Bulk', 'General Cargo', 'Roll-on/roll-off', 'Tanker']

fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, ship_type in enumerate(ship_types):
    normal_data = arrivals_df[~arrivals_df['is_crisis']][ship_type]
    crisis_data = arrivals_df[arrivals_df['is_crisis']][ship_type]
    
    axes[idx].hist([normal_data, crisis_data], bins=30, label=['Normal', 'Crisis'], alpha=0.7)
    axes[idx].set_title(f'{ship_type} Distribution', fontsize=14, fontweight='bold')
    axes[idx].set_xlabel('Daily Arrivals')
    axes[idx].set_ylabel('Frequency')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

axes[-1].hist([arrivals_df[~arrivals_df['is_crisis']]['Total'], 
               arrivals_df[arrivals_df['is_crisis']]['Total']], 
              bins=30, label=['Normal', 'Crisis'], alpha=0.7)
axes[-1].set_title('Total Traffic Distribution', fontsize=14, fontweight='bold')
axes[-1].set_xlabel('Daily Arrivals')
axes[-1].set_ylabel('Frequency')
axes[-1].legend()
axes[-1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Time Series: Ship Types Over Time with Crisis Periods

In [ ]:
fig, ax = plt.subplots(figsize=(22, 10))

for ship_type in ship_types:
    ax.plot(arrivals_df.index, arrivals_df[ship_type], label=ship_type, linewidth=2, alpha=0.8)

for _, crisis in crisis_df.iterrows():
    ax.axvspan(crisis['start'], crisis['end'], alpha=0.2, color='red')
    mid_point = crisis['start'] + (crisis['end'] - crisis['start']) / 2
    ax.text(mid_point, ax.get_ylim()[1] * 0.95, crisis['event'], 
            rotation=90, verticalalignment='top', fontsize=9, alpha=0.7)

ax.set_title('Ship Arrivals by Type Over Time (Red = Crisis Periods)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Daily Arrivals', fontsize=12)
ax.legend(loc='upper left', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(22, 10))

ax.plot(arrivals_df.index, arrivals_df['Total'], linewidth=2.5, color='navy', label='Total Traffic')
ax.plot(arrivals_df.index, arrivals_df['7-day Moving Average'], 
        linewidth=2, color='orange', linestyle='--', label='7-Day MA', alpha=0.8)

for _, crisis in crisis_df.iterrows():
    ax.axvspan(crisis['start'], crisis['end'], alpha=0.25, color='red')

ax.set_title('Total Ship Traffic with Crisis Periods Highlighted', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Daily Arrivals', fontsize=12)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Analysis by Crisis Event

In [ ]:
crisis_analysis = []

for _, crisis in crisis_df.iterrows():
    crisis_data = arrivals_df[(arrivals_df.index >= crisis['start']) & (arrivals_df.index <= crisis['end'])]
    
    pre_crisis_start = crisis['start'] - timedelta(days=30)
    pre_crisis_data = arrivals_df[(arrivals_df.index >= pre_crisis_start) & (arrivals_df.index < crisis['start'])]
    
    for ship_type in ship_types + ['Total']:
        crisis_analysis.append({
            'Event': crisis['event'],
            'Sentiment': crisis['sentiment'],
            'Ship_Type': ship_type,
            'Pre_Crisis_Avg': pre_crisis_data[ship_type].mean(),
            'Crisis_Avg': crisis_data[ship_type].mean(),
            'Pct_Change': ((crisis_data[ship_type].mean() - pre_crisis_data[ship_type].mean()) / 
                          pre_crisis_data[ship_type].mean() * 100),
            'Days': len(crisis_data)
        })

crisis_analysis_df = pd.DataFrame(crisis_analysis)
crisis_analysis_df = crisis_analysis_df.round(2)
crisis_analysis_df

In [ ]:
tanker_impact = crisis_analysis_df[crisis_analysis_df['Ship_Type'] == 'Tanker'].sort_values('Pct_Change')
tanker_impact

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, ship_type in enumerate(ship_types + ['Total']):
    data = crisis_analysis_df[crisis_analysis_df['Ship_Type'] == ship_type]
    
    x = range(len(data))
    colors = ['red' if val < 0 else 'green' for val in data['Pct_Change']]
    
    axes[idx].bar(x, data['Pct_Change'], color=colors, alpha=0.7)
    axes[idx].set_xticks(x)
    axes[idx].set_xticklabels(data['Event'], rotation=45, ha='right', fontsize=8)
    axes[idx].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
    axes[idx].set_title(f'{ship_type} Impact by Crisis', fontsize=12, fontweight='bold')
    axes[idx].set_ylabel('% Change from Pre-Crisis')
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Sentiment-Based Analysis

In [ ]:
sentiment_stats = arrivals_df.groupby('sentiment')[ship_types + ['Total']].agg(['mean', 'std', 'count'])
sentiment_stats

In [ ]:
sentiment_order = ['Stable', 'Elevated Tension', 'High Tension', 'Supply Concerns', 'Extreme Tension', 'Critical Disruption']
sentiment_means = arrivals_df.groupby('sentiment')[ship_types].mean().reindex(sentiment_order)

fig, ax = plt.subplots(figsize=(18, 10))

sentiment_means.T.plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Average Daily Arrivals by Ship Type and Sentiment', fontsize=16, fontweight='bold')
ax.set_xlabel('Ship Type', fontsize=12)
ax.set_ylabel('Average Daily Arrivals', fontsize=12)
ax.legend(title='Sentiment', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## Year-over-Year Comparison

In [ ]:
arrivals_df['Year'] = arrivals_df.index.year
arrivals_df['Month'] = arrivals_df.index.month

yearly_avg = arrivals_df.groupby('Year')[ship_types + ['Total']].mean()
yearly_avg

In [ ]:
fig, ax = plt.subplots(figsize=(18, 10))

yearly_avg[ship_types].plot(kind='bar', ax=ax, width=0.8)
ax.set_title('Average Daily Arrivals by Ship Type and Year', fontsize=16, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Average Daily Arrivals', fontsize=12)
ax.legend(title='Ship Type', bbox_to_anchor=(1.05, 1), loc='upper left')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## Most Affected Ship Types

In [ ]:
ship_type_impact = crisis_analysis_df.groupby('Ship_Type')['Pct_Change'].agg(['mean', 'min', 'max', 'std'])
ship_type_impact = ship_type_impact.sort_values('mean')
ship_type_impact

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

colors = ['red' if val < 0 else 'green' for val in ship_type_impact['mean']]
ax.barh(ship_type_impact.index, ship_type_impact['mean'], color=colors, alpha=0.7)
ax.axvline(x=0, color='black', linestyle='-', linewidth=1)
ax.set_title('Average Impact on Ship Types During Crisis Periods', fontsize=16, fontweight='bold')
ax.set_xlabel('Average % Change from Pre-Crisis Levels', fontsize=12)
ax.set_ylabel('Ship Type', fontsize=12)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## Recovery Time Analysis

In [ ]:
recovery_analysis = []

for _, crisis in crisis_df.iterrows():
    crisis_data = arrivals_df[(arrivals_df.index >= crisis['start']) & (arrivals_df.index <= crisis['end'])]
    
    post_crisis_end = crisis['end'] + timedelta(days=60)
    post_crisis_data = arrivals_df[(arrivals_df.index > crisis['end']) & (arrivals_df.index <= post_crisis_end)]
    
    pre_crisis_start = crisis['start'] - timedelta(days=30)
    pre_crisis_data = arrivals_df[(arrivals_df.index >= pre_crisis_start) & (arrivals_df.index < crisis['start'])]
    
    if len(post_crisis_data) > 0:
        recovery_analysis.append({
            'Event': crisis['event'],
            'Crisis_Duration_Days': len(crisis_data),
            'Pre_Crisis_Avg': pre_crisis_data['Total'].mean(),
            'Crisis_Avg': crisis_data['Total'].mean(),
            'Post_Crisis_Avg': post_crisis_data['Total'].mean(),
            'Recovery_Pct': ((post_crisis_data['Total'].mean() - crisis_data['Total'].mean()) / 
                            (pre_crisis_data['Total'].mean() - crisis_data['Total'].mean()) * 100)
        })

recovery_df = pd.DataFrame(recovery_analysis).round(2)
recovery_df

## Key Findings Summary

In [ ]:
findings = {
    'Total Crisis Days': arrivals_df['is_crisis'].sum(),
    'Total Normal Days': (~arrivals_df['is_crisis']).sum(),
    'Avg Daily Traffic (Normal)': arrivals_df[~arrivals_df['is_crisis']]['Total'].mean().round(2),
    'Avg Daily Traffic (Crisis)': arrivals_df[arrivals_df['is_crisis']]['Total'].mean().round(2),
    'Overall Impact (%)': ((arrivals_df[arrivals_df['is_crisis']]['Total'].mean() - 
                           arrivals_df[~arrivals_df['is_crisis']]['Total'].mean()) / 
                          arrivals_df[~arrivals_df['is_crisis']]['Total'].mean() * 100).round(2),
    'Most Affected Ship Type': ship_type_impact.index[0],
    'Least Affected Ship Type': ship_type_impact.index[-1]
}

findings_df = pd.DataFrame([findings]).T
findings_df.columns = ['Value']
findings_df